# From the LHC to the High-Luminosity LHC: top-quark pairs with 200 collisions per crossing

<CENTER><img src="https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/master/images/ATLASOD.gif?raw=1" style="width:50%"></CENTER>

This notebook uses ATLAS Open Data http://opendata.atlas.cern to compare top-quark pair
($t\bar{t}$) events in today's ATLAS detector with the same kind of events simulated for the
**High-Luminosity LHC (HL-LHC)**.

ATLAS Open Data provides open access to proton-proton collision data at the LHC for educational
purposes. ATLAS Open Data resources are ideal for high-school, undergraduate and postgraduate
students.

Notebooks are web applications that allow you to create and share documents that can contain for
example:
1. live code
2. visualisations
3. narrative text

### What is the HL-LHC?
The HL-LHC is the next phase of the Large Hadron Collider, planned to start around 2030. It will
collect about 3000 fb$^{-1}$ of collisions, roughly twenty times the 140 fb$^{-1}$ recorded in
Run 2 (2015–2018). More collisions mean more chances to see rare processes, but they come at a
price: **pile-up**. In every bunch crossing about 200 proton-proton collisions happen at the same
time, instead of about 34 on average in Run 2. Only one of them is the collision we want to
study; the others add extra particles all over the detector.

To cope with this, ATLAS is being upgraded. The most important new parts are:
* the **Inner Tracker (ITk)**, an all-silicon tracker that measures charged particles out to
  $|\eta| = 4$, instead of $|\eta| = 2.5$ today;
* the **High-Granularity Timing Detector (HGTD)**, which measures the time of particles in the
  forward region, to separate collisions that happen close together;
* new trigger and readout electronics for the calorimeters and the muon system.

### Our two samples
We compare two simulated samples of single-lepton $t\bar{t}$ events: one of the two W bosons
decays to a lepton and a neutrino, the other to two quarks. Both were generated with the same
programs (Powheg + Pythia 8) and processed with the same analysis software and object
definitions, so the differences we see come from the detector and from pile-up.

| | Run 2 | HL-LHC |
|---|---|---|
| Centre-of-mass energy | 13 TeV | 13.6 TeV |
| Collisions per bunch crossing | about 24 (2015–2016 conditions) | about 200 |
| Detector | ATLAS in Run 2 | ATLAS after the upgrade, with the ITk |

In this notebook we will look at:
1. **Pile-up**: how many collisions there are, and how many we can see.
2. **What pile-up does**: extra jets, and a less precise missing transverse momentum.

### Running a Jupyter notebook
A Jupyter notebook consists of cell blocks,
    each containing lines of Python code.
Each cell can be run independently of each other,
    yielding respective outputs below the cells.
Conventionally,
    cells are run in order from top to bottom.

- To run the whole notebook, in the top menu click Cell $\to$ Run All.

- To propagate a change you've made to a piece of code, click Cell $\to$ Run All Below.

- You can also run a single code cell, by clicking Cell $\to$ Run Cells, or using the keyboard shortcut Shift+Enter.

For more information,
    refer to [here](https://www.codecademy.com/article/how-to-use-jupyter-notebooks).

## ATLAS Open Data Initialisation

### First time package installation on your computer (not needed on mybinder)
This first cell installs the required python packages.
It only needs to be run the first time you open this notebook on your computer.
If you close Jupyter and re-open on the same computer, you won't need to run this first cell again.

If this is opened on mybinder, you don't need to run this cell.

In [ ]:
import sys
!pip install atlasopenmagic
from atlasopenmagic import install_from_environment
install_from_environment()

### To setup everytime
We're going to import a number of packages to help us:
* `numpy`: provides numerical calculations such as histogramming
* `matplotlib`: common tool for making plots, figures, images, visualisations
* `uproot`: processes `.root` files typically used in particle physics into data formats used in python
* `awkward`: introduces `awkward` arrays, a format that generalizes `numpy` to nested data with possibly variable length lists
* `atlasopenmagic`: finds information about the ATLAS Open Data samples for us

In [ ]:
import numpy as np # for numerical calculations such as histogramming
import matplotlib.pyplot as plt # for plotting
from matplotlib.ticker import AutoMinorLocator, MaxNLocator # for minor ticks and tick spacing
import uproot # for reading .root files
import awkward as ak # to represent nested data in columnar format
import atlasopenmagic as atom # to find information about ATLAS Open Data samples
%config InlineBackend.figure_formats = ['svg'] # sharp vector plots, at any zoom

## Where's my data?

The Run 2 sample is the $t\bar{t}$ single-lepton sample (dataset number 601497) of the ATLAS
Open Data for research. The HL-LHC sample was simulated with the upgraded detector. Both were
converted with the same settings into small files with one entry per event, which we read from
`RUN2_FILE` and `HLLHC_FILE`. 10,000 events of each sample are plenty for this notebook.

We use `atlasopenmagic` to look up the description of the Run 2 sample.

In [ ]:
RUN2_FILE = ('/global/cfs/cdirs/atlas/mironova/open_data/OpenData/run/'
             'run2_601497_10k/output.root') # Run 2 ttbar
HLLHC_FILE = ('/global/cfs/cdirs/atlas/mironova/open_data/OpenData/run/'
              'phase2_matched_10k/output.root') # HL-LHC ttbar
N_EVENTS = 10_000 # number of events to read from each sample

atom.set_release('2024r-pp') # ATLAS Open Data for research
run2_info = atom.get_metadata(601497)
print(run2_info['physics_short'], '-', run2_info['description'])

### Reading the samples
Both files contain the same variables. We read the jets, the missing transverse momentum, the
number of collisions per bunch crossing (`mu`) and the number of reconstructed collision vertices
(`npv`). Energies and momenta are stored in MeV, so we convert them to GeV.

In simulation we know where every jet comes from: each jet has a flag `hs` that says whether it
belongs to the $t\bar{t}$ collision ("hard scatter") or to one of the pile-up collisions.

An electron leaves its energy in the calorimeter just like a jet does, so every electron is also
reconstructed as a jet. We remove these duplicates: jets closer than $\Delta R = 0.2$ to an
electron, where $\Delta R = \sqrt{\Delta\eta^2 + \Delta\phi^2}$ is the distance between them.

In [ ]:
def remove_electron_jets(jets, electrons):
    pairs = ak.cartesian({'jet': jets, 'el': electrons}, nested=True) # every jet with every electron
    d_phi = np.arctan2(np.sin(pairs.jet.phi - pairs.el.phi), np.cos(pairs.jet.phi - pairs.el.phi))
    d_r = np.sqrt((pairs.jet.eta - pairs.el.eta)**2 + d_phi**2)
    return jets[~ak.any(d_r < 0.2, axis=2)] # keep jets that are far from all electrons


def load(path, n_events):
    a = uproot.open(path)['analysis'].arrays(entry_stop=n_events)
    GeV = 1000 # the files store energies in MeV
    electrons = ak.zip({'eta': a.elec_eta, 'phi': a.elec_phi})
    jet = ak.zip({'pt': a.jet_pt_NOSYS / GeV, 'eta': a.jet_eta, 'phi': a.jet_phi,
                  'hs': a.jet_isJvtHS == 1})
    return ak.zip({
        'jet': remove_electron_jets(jet, electrons),
        'met_x': a.met_mpx / GeV, 'met_y': a.met_mpy / GeV,
        'truth_met_x': a.truth_met_mpx / GeV, 'truth_met_y': a.truth_met_mpy / GeV,
        'mu': a.averageInteractionsPerCrossing,
        'npv': ak.sum(a.pv_type != 0, axis=1), # vertex type 0 is a placeholder, not a vertex
    }, depth_limit=1)


run2 = load(RUN2_FILE, N_EVENTS)
hllhc = load(HLLHC_FILE, N_EVENTS)
print(f'Run 2: {len(run2)} events, HL-LHC: {len(hllhc)} events')
hllhc.jet.pt[:3].tolist() # jet pT of the first three HL-LHC events

### Making plots
We define a few helper functions for our plots. `atlas_style` puts ticks on all four sides, adds
minor ticks and labels the plot. `compare` draws distributions from several samples on top of
each other and adds a lower panel with the ratio of each sample to the first one in the legend.
Each distribution is **normalised to unit area**, so that we compare their shapes. The shaded
bands show the statistical uncertainty. `stack` draws the hard-scatter and pile-up jets of one
sample on top of each other.

In [ ]:
RUN2 = {'label': 'Run 2, 13 TeV', 'color': '#8b008b', 'linestyle': '-'} # dark magenta
HLLHC = {'label': r'HL-LHC, 13.6 TeV, $\langle\mu\rangle$ = 200', 'color': '#228b22', 'linestyle': '--'} # forest green


def atlas_style(ax, xlabel=None, ylabel=None, text=()):
    if ax.get_xscale() == 'linear':
        ax.xaxis.set_minor_locator(AutoMinorLocator()) # minor ticks on the x axis
    ax.yaxis.set_minor_locator(AutoMinorLocator()) # minor ticks on the y axis
    ax.tick_params(which='both', direction='in', top=True, right=True) # ticks inside, on all sides
    if xlabel:
        ax.set_xlabel(xlabel, x=1, horizontalalignment='right')
    if ylabel:
        ax.set_ylabel(ylabel, y=1, horizontalalignment='right')
    if text is not None: # text=None for the ratio panel
        lines = [('ATLAS Open Data', {'fontsize': 14}), ('for education', {'fontsize': 11, 'style': 'italic'})]
        for i, (line, kwargs) in enumerate(lines + [(t, {'fontsize': 11}) for t in text]):
            ax.text(0.04, 0.92 - 0.065 * i, line, transform=ax.transAxes, **kwargs)


def band(ax, bins, low, high, color):
    """Shaded band following a histogram; bins with nan are left empty."""
    ax.fill_between(bins, np.append(low, low[-1]), np.append(high, high[-1]), step='post',
                    color=color, alpha=0.2, linewidth=0)


def compare(samples, bins, xlabel, ylabel='Normalised to unit area', text=()):
    """samples: list of (values, style). The lower panel shows each sample's ratio to the first."""
    fig, (ax, rax) = plt.subplots(2, 1, figsize=(8, 6.5), sharex=True,
                                  gridspec_kw={'height_ratios': [3, 1], 'hspace': 0})
    shapes = []
    for values, style in samples:
        counts, _ = np.histogram(ak.to_numpy(ak.flatten(ak.Array(values), axis=None)), bins)
        norm = counts.sum() * np.diff(bins) # unit area
        y, err = counts / norm, np.sqrt(counts) / norm # height and statistical uncertainty
        ax.stairs(y, bins, color=style['color'], linestyle=style['linestyle'], linewidth=2,
                  label=style['label'])
        band(ax, bins, y - err, y + err, style['color'])
        shapes.append((y, err))
    reference = np.where(shapes[0][0] > 0, shapes[0][0], np.nan) # no ratio where the first is empty
    for (y, err), (_, style) in zip(shapes[1:], samples[1:]):
        rax.stairs(y / reference, bins, color=style['color'], linestyle=style['linestyle'], linewidth=2,
                   baseline=None) # no vertical lines down to zero at the edges
        band(rax, bins, (y - err) / reference, (y + err) / reference, style['color'])
    rax.axhline(1, color='grey', linewidth=1)
    rax.set_ylim(0.5, 1.5)
    ax.set_xlim(bins[0], bins[-1])
    ax.set_ylim(0, 1.5 * max(np.max(y + e) for y, e in shapes)) # room for the labels
    ax.legend(frameon=False, loc='upper right')
    atlas_style(ax, ylabel=ylabel, text=text)
    atlas_style(rax, xlabel, 'Ratio', text=None)
    ax.yaxis.set_major_locator(MaxNLocator(prune='lower')) # keep the panels' tick labels apart
    rax.yaxis.set_major_locator(MaxNLocator(nbins=4, prune='upper'))
    return ax, rax


def stack(ax, events, variable, bins, style):
    """Hard-scatter and pile-up jets of one sample, per event, stacked; returns the highest bin."""
    jets = events.jet
    hard, _ = np.histogram(ak.to_numpy(ak.flatten(variable(jets[jets.hs]))), bins)
    pileup, _ = np.histogram(ak.to_numpy(ak.flatten(variable(jets[~jets.hs]))), bins)
    n = len(events)
    ax.stairs(hard / n, bins, fill=True, color=style['color'] + '80', label='hard-scatter jets') # '80': half-transparent
    ax.stairs((hard + pileup) / n, bins, baseline=hard / n, fill=True, color='#bdbdbd', label='pile-up jets')
    ax.set_xlim(bins[0], bins[-1])
    ax.legend(frameon=False, loc='upper right')
    return np.max((hard + pileup) / n)

## 1. Pile-up
$\mu$ is the average number of proton-proton collisions per bunch crossing. Only one of them is
the $t\bar{t}$ event we are interested in; the rest are mostly soft collisions. The Run 2 sample
follows the conditions of 2015–2016, with about 24 collisions on average; the HL-LHC sample has
between 190 and 210.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for events, style in ((run2, RUN2), (hllhc, HLLHC)):
    ax.hist(ak.to_numpy(events.mu), bins=np.arange(0, 232, 2), histtype='stepfilled', linewidth=1.5,
            facecolor=style['color'] + '4d', edgecolor=style['color'], label=style['label']) # '4d': 30% opaque fill
ax.set_xlim(0, 230)
ax.set_ylim(0, 1.6 * ax.get_ylim()[1]) # room for the labels
ax.legend(frameon=False, loc='upper right')
atlas_style(ax, r'Average interactions per bunch crossing, $\mu$', 'Events / 2')
plt.show()

Each collision should leave a **primary vertex**: a point on the beam line where several tracks
meet. In both samples fewer vertices are reconstructed than there are collisions, because some
collisions produce too few tracks. At the HL-LHC only about half of the collisions give a vertex:
many happen so close to each other that their vertices merge. This is why the HGTD will add a time
measurement, to tell apart collisions that overlap in space.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for events, style in ((run2, RUN2), (hllhc, HLLHC)):
    ax.hist(ak.to_numpy(events.npv), bins=np.arange(0, 162, 2), histtype='stepfilled', linewidth=1.5,
            facecolor=style['color'] + '4d', edgecolor=style['color'], label=style['label']) # '4d': 30% opaque fill
ax.set_ylim(0, 1.6 * ax.get_ylim()[1]) # room for the labels
ax.legend(frameon=False, loc='upper right')
atlas_style(ax, 'Number of reconstructed primary vertices', 'Events / 2')
plt.show()
for name, events in (('Run 2', run2), ('HL-LHC', hllhc)):
    print(f'{name}: on average {np.mean(events.npv):.0f} vertices for {np.mean(events.mu):.0f} collisions')

## 2. What pile-up does

### 2.1 Extra jets
Pile-up collisions produce extra jets that have nothing to do with the $t\bar{t}$ event. Using the
`hs` flag we can see which of the jets come from pile-up. Let's first look at the transverse
momentum of the jets in the central part of the detector, $|\eta| < 2.5$.

In [ ]:
bins = np.linspace(20, 100, 33)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
central_pt = lambda jets: jets.pt[abs(jets.eta) < 2.5] # pT of the central jets
highest = max(stack(ax, events, central_pt, bins, style)
              for ax, events, style in zip(axes, (run2, hllhc), (RUN2, HLLHC)))
for ax, style in zip(axes, (RUN2, HLLHC)):
    atlas_style(ax, r'Jet $p_T$ [GeV]', r'Jets with $|\eta| < 2.5$ per event' if ax is axes[0] else None,
                text=[style['label']])
axes[0].set_ylim(0, 1.6 * highest) # room for the labels
plt.show()

Most pile-up jets are soft: they pile up at low $p_T$, while the jets from the $t\bar{t}$ decay
reach much higher. At the HL-LHC there are many more of them than in Run 2.

Where in the detector do the pile-up jets end up? The next plot shows all jets as a function of
$\eta$. The shaded area is the forward region, beyond $|\eta| = 2.5$, where today's tracker does
not reach.

In [ ]:
ETA_PT_MIN = 20 # GeV, try 30 or 50!
bins = np.linspace(-4.5, 4.5, 37)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
eta_above = lambda jets: jets.eta[jets.pt > ETA_PT_MIN] # eta of the jets above the threshold
highest = max(stack(ax, events, eta_above, bins, style)
              for ax, events, style in zip(axes, (run2, hllhc), (RUN2, HLLHC)))
for ax, style in zip(axes, (RUN2, HLLHC)):
    for sign in (-1, 1):
        ax.axvspan(sign * 2.5, sign * 4.5, color='grey', alpha=0.12, zorder=0) # no tracks today
    atlas_style(ax, r'Jet $\eta$', 'Jets per event' if ax is axes[0] else None,
                text=[style['label'], f'$p_T > {ETA_PT_MIN}$ GeV'])
axes[0].set_ylim(0, 1.7 * highest) # room for the labels
plt.show()
for name, events in (('Run 2', run2), ('HL-LHC', hllhc)):
    jets = events.jet[events.jet.pt > ETA_PT_MIN]
    pileup = jets[~jets.hs]
    print(f'{name}: {ak.sum(~jets.hs) / len(events):.2f} pile-up jets per event, '
          f'{ak.mean(ak.flatten(abs(pileup.eta) > 2.5)):.0%} of them with |eta| > 2.5')

At the HL-LHC most pile-up jets are **forward**: beyond $|\eta| = 2.5$ they are common, and beyond
$|\eta| = 4$ almost every jet comes from pile-up. Today's tracker stops at $|\eta| = 2.5$, so these
jets have no tracks that could tell us which collision they come from. Identifying them with the
tracks of the ITk and the timing of the HGTD is one of the main goals of the upgrade.

In the central region there is a simpler way to get rid of most pile-up jets: **only keep jets
above a $p_T$ threshold**. The next plot shows what fraction of the central jets above a threshold
still come from pile-up.

In [ ]:
THRESHOLDS = np.arange(20, 101, 5) # GeV, try other values!


def pileup_fraction(events):
    jets = events.jet[abs(events.jet.eta) < 2.5] # the central part of the detector
    fraction = []
    for t in THRESHOLDS:
        above = jets[jets.pt > t]
        fraction.append(1 - ak.mean(ak.flatten(above.hs))) # pile-up jets / all jets
    return fraction


fig, ax = plt.subplots(figsize=(8, 5))
for events, style, marker in ((run2, RUN2, 'o-'), (hllhc, HLLHC, 's-')):
    ax.plot(THRESHOLDS, pileup_fraction(events), marker, color=style['color'], label=style['label'])
ax.set_ylim(0, 0.3)
ax.legend(frameon=False, loc='upper right')
atlas_style(ax, r'Jet $p_T$ threshold [GeV]', r'Fraction of pile-up jets, $|\eta| < 2.5$')
plt.show()
for events, name in ((run2, 'Run 2'), (hllhc, 'HL-LHC')):
    print(f'{name}: ' + ', '.join(f'{f:.1%} above {t} GeV' for t, f in zip(THRESHOLDS, pileup_fraction(events)) if t in (20, 30, 40)))

Already a threshold of 30 GeV removes most pile-up jets in the central region: at the HL-LHC their
fraction drops from about one in six to about one in twenty. The price is that we also lose the
softest jets from the $t\bar{t}$ decay. The upgraded detector will help to identify pile-up jets
without such a hard cut, using tracks and timing.

### 2.2 Missing transverse momentum
Neutrinos escape the detector without leaving a trace. We find them from the imbalance of
momentum in the plane transverse to the beam, the missing transverse momentum
$E_T^{\mathrm{miss}}$. In simulation we know the true neutrino momentum, so we can compare each
event's measured $E_x^{\mathrm{miss}}$ and $E_y^{\mathrm{miss}}$ with the truth: the narrower this
distribution, the better the measurement (the **resolution**).

In [ ]:
def met_residuals(events):
    return np.concatenate([ak.to_numpy(events.met_x - events.truth_met_x),
                           ak.to_numpy(events.met_y - events.truth_met_y)])


res_run2, res_hl = met_residuals(run2), met_residuals(hllhc)
ax, rax = compare([(res_run2, RUN2), (res_hl, HLLHC)], np.linspace(-100, 100, 41),
                  r'$E_{x,y}^{\mathrm{miss}}$ (measured) $-$ $E_{x,y}^{\mathrm{miss}}$ (truth) [GeV]',
                  text=[r'single-lepton $t\bar{t}$'])
rax.set_ylim(0, 3)
plt.show()
print(f'Resolution (RMS): Run 2 {np.std(res_run2):.1f} GeV, HL-LHC {np.std(res_hl):.1f} GeV')

Every pile-up collision adds random energy to the event. Because both samples store the number of
vertices, we can follow the resolution from Run 2 conditions to the HL-LHC: it gets worse as the
number of vertices grows.

In [ ]:
def resolution_vs_npv(events, n_bins=5):
    npv = ak.to_numpy(events.npv)
    dx = ak.to_numpy(events.met_x - events.truth_met_x)
    edges = np.quantile(npv, np.linspace(0, 1, n_bins + 1)) # bins with the same number of events
    centres, rms, err = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        in_bin = (npv >= lo) & (npv <= hi)
        centres.append(npv[in_bin].mean())
        rms.append(dx[in_bin].std())
        err.append(dx[in_bin].std() / np.sqrt(2 * in_bin.sum())) # uncertainty on an RMS
    return centres, rms, err


fig, ax = plt.subplots(figsize=(8, 5))
for events, style, marker in ((run2, RUN2, 'o-'), (hllhc, HLLHC, 's-')):
    centres, rms, err = resolution_vs_npv(events)
    ax.errorbar(centres, rms, yerr=err, fmt=marker, color=style['color'], label=style['label'])
ax.set_xlim(0, 130)
ax.set_ylim(0, 1.6 * ax.get_ylim()[1]) # room for the labels
ax.legend(frameon=False, loc='upper right')
atlas_style(ax, 'Number of reconstructed primary vertices',
            r'RMS of $E_x^{\mathrm{miss}}$ (measured $-$ truth) [GeV]')
plt.show()

## Summary

| | What we saw |
|---|---|
| Pile-up | About 200 collisions per bunch crossing instead of 24, but only about 100 reconstructed vertices |
| Jets | Many extra soft jets from pile-up, most of them in the forward region; in the central region a $p_T$ cut of 30 GeV removes most of them |
| $E_T^{\mathrm{miss}}$ | The resolution gets worse as the number of vertices grows, from Run 2 to the HL-LHC |

### Try it yourself!
1. Change `ETA_PT_MIN` in Section 2.1: how high would a cut on the forward jets have to be?
2. Change `THRESHOLDS` in Section 2.1: which threshold keeps the central HL-LHC pile-up fraction below 2%?

<div class="alert alert-block alert-info">
We welcome your feedback on this notebook or any of our other materials! Please <a href="https://forms.gle/zKBqS1opAHHemv9U7">fill out this survey</a> to let us know how we're doing, and you can enter a raffle to win some <a href="https://atlas-secretariat.web.cern.ch/merchandise">ATLAS merchandise</a>!
</div>